# Transformers and Context - Concepts

**Business problem:** use word order and context before suggesting categories for a job publisher.

A model supports a decision; it does not make the business decision for us.

## Learning objectives

- Explain token positions and one attention update.
- Distinguish BERT features from category predictions.
- Contrast masked-token training and next-token generation.

## Dataset and practical scope

Token/vector examples and decoding probabilities in this concept notebook are constructed. No Transformer is trained here.

The companion [exercise](transformers_exercise.ipynb) and [solution](transformers_solution.ipynb) use the **5,011 real redacted job ads** behind Week 08 page 37, not synthetic job advertisements. They compare frequency, TF-IDF and frozen multilingual BERT on the original chronological partitions.

All code here is supplied. Run the cells from top to bottom.

## Libraries and settings

Use the prepared Python 3.11 Codespace kernel. No GPU or API key is needed.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 7
np.random.seed(SEED)
DATA_ROOT = Path('../../../00_Data')

## 1. Baseline: a static mean loses order

The same word multiset gives the same static mean. Contextual models can use token-position pairings before pooling.

In [2]:
first = ['engineer', 'supports', 'customers']
second = ['customers', 'supports', 'engineer']
display(pd.DataFrame({'position': [1, 2, 3], 'sentence A': first, 'sentence B': second}))

,position,sentence A,sentence B
0,1,engineer,customers
1,2,supports,supports
2,3,customers,engineer


## 2. One self-attention update

Hold one token position fixed. Scores for the other positions become softmax weights; multiply each by its contributed value vector and add. A head has its own learned transformations. Attention weights are computed for each input, not stored as one permanent table or used as causal explanations.

In [3]:
scores = np.array([2., 1., 0.])
weights = np.exp(scores - scores.max())
weights /= weights.sum()
values = np.array([[1., 0.], [0., 1.], [1., 1.]])
print('Constructed weights:', weights.round(3))
print('New vector for the selected position:', (weights @ values).round(3))

Constructed weights: [0.665 0.245 0.09 ]
New vector for the selected position: [0.755 0.335]


## 3. BERT: contextual features before a new classifier

BERT means Bidirectional Encoder Representations from Transformers. Masked language modelling hides tokens and predicts their original identities using context on both sides. The targets come from text. In the exercise, BERT stays frozen; a new classifier learns job categories from pooled features.

In [4]:
token_features = np.array([[1., 0., 2.], [0., 2., 1.], [2., 1., 0.]])
pooled = token_features.mean(axis=0)
print('Constructed token matrix shape:', token_features.shape)
print('Pooled feature shape:', pooled.shape, '| Values:', pooled)

Constructed token matrix shape: (3, 3)
Pooled feature shape: (3,) | Values: [1. 1. 1.]


## 4. Encoder versus decoder

An encoder can use both-side input context. An encoder-decoder also generates with access to encoded source text. A GPT-style decoder-only model uses causal context and predicts the next token. Training uses known next tokens; generation appends the selected token and repeats.

In [5]:
prefix = ['We', 'need', 'a', 'software']
vocabulary = ['engineer', 'analyst', '.']
next_probabilities = np.array([0.7, 0.2, 0.1])
chosen = vocabulary[int(np.argmax(next_probabilities))]
print('Constructed greedy continuation:', ' '.join(prefix + [chosen]))

Constructed greedy continuation: We need a software engineer


## Evaluation and interpretation

These calculations check a mechanism, not language understanding. Compare the real frozen-feature classifier with frequency and TF-IDF baselines in the exercise. Fluent continuation is not factual correctness.

## Summary and business implications

A feature vector is not a category probability. Pooling changes output shape, not the training objective. Frozen parameters can still produce input-dependent vectors.

## Limitations

This is a small teaching example, not evidence of deployment performance. Keep evaluation separate from fitting and model selection. Check errors and the user's decision, not only a headline metric.

## Further exercises

- Explain why attention alone is not a complete Transformer.
- Does adding pooling make BERT retrieval-trained?
- Why does prompting normally not update model weights?

## References

**Required reading:** BERT overview: https://huggingface.co/docs/transformers/model_doc/bert
**Instructor references:** Vaswani et al. (2017), *Attention Is All You Need*, https://arxiv.org/abs/1706.03762;
Devlin et al. (2019), *BERT*, https://aclanthology.org/N19-1423/;
Brown et al. (2020), *Language Models are Few-Shot Learners*, https://arxiv.org/abs/2005.14165.

### Jupyter notebook - footer info

Include your name, submission date and a short description of your changes when submitting.